In [ ]:
import os
import sys
import math
import torch
import random
import numpy as np
import torch.nn as nn
from pathlib import Path
import torch.nn.functional as F
import matplotlib.pyplot as plt
from einops import rearrange, reduce, repeat, einsum


import training_engine  
import inference_engine
from utils import RMSNorm, init_custom_weight



MODEL_CONFIGS = {
    "embed_dim"   : 2**5,   # 32
    "max_seq_len" : 2**6,   # 64
    "vocab_size"  : 2**11,  # 2048
    "num_attn_heads": 1,
    "num_layers": 3,
   
    "batch_size"  : 2**4,   # 16
    "learning_rate" : 1e-3,

    "save_path" : "../models/level3_single_head_attn.pt",
    "loss_history" : "../models/level3_single_head_attn_loss_history.txt",

}



# Device configuration
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)




cuda


In [ ]:
class Level3SingleHeadCausalAttention_Model(nn.Module):

    # embed, PE, blocks, final norm, unembed

    def __init__(self, config=MODEL_CONFIGS):
        super().__init__()
        ...


    def forward():
        ...

In [ ]:
class Level3Transformer_Block(nn.Module):

    # RMS norm, attention, residual

    def __init__(self, config):
        super().__init__()
        ...


    def forward():
        ...


In [ ]:
class SingleHeadCausalAttention_SubBlock(nn.Module):

    # if kv_cache is not None -> x:[B, 1, D] decode
    # if kv_cache is     None -> x:[B, T, D] prefill

    def __init__(self, config):
        super().__init__()

        self.embed_dim = config["embed_dim"]
        self.num_attn_heads = config["num_attn_heads"]
        self.head_dim = self.embed_dim // self.num_attn_heads

        assert self.embed_dim % self.num_attn_heads == 0


        # Projections: embed_dim → num_heads * head_dim
        self.query_w = nn.Parameter(torch.empty(self.embed_dim, self.num_attn_heads * self.head_dim))
        self.key_w   = nn.Parameter(torch.empty(self.embed_dim, self.num_attn_heads * self.head_dim))
        self.value_w = nn.Parameter(torch.empty(self.embed_dim, self.num_attn_heads * self.head_dim))

        # Output projection: num_heads * head_dim → embed_dim
        self.out_w   = nn.Parameter(torch.empty(self.num_attn_heads * self.head_dim, self.embed_dim))


        # Initialize parameters
        self.reset_parameters()



    def reset_parameters(self, config):

        # 1. Standard Projections
        init_custom_weight(self.query_w, is_residual_output=False)
        init_custom_weight(self.key_w,   is_residual_output=False)
        init_custom_weight(self.value_w, is_residual_output=False)

        # 2. Residual Output Projections (Scaled by depth)
        init_custom_weight(self.W_out, is_residual_output=True, num_layers=config["num_layers"])



    def forward(
        self,
        x: torch.tensor,
        use_cache: bool = False,
        kv_cache: tuple[torch.Tensor, torch.Tensor] | None = None, 
    ):
        
        batch_size, seq_len, embed_dim = x.shape 


        # -------------------------------------------------
        # 1. Linear projections (via einops.einsum)
        #    (batch_size, seq_len, embed_dim) @ Wᵀ
        # →  (batch_size, seq_len, num_attn_heads * head_dim)
        #
        # nn.Linear.weight has shape (out_features, in_features), 
        # so the einsum contracts on embed_dim.
        # -------------------------------------------------
        q_vec = einsum(
            x, self.query_w.weight,
            "batch_size seq_len embed_dim, "
            "num_attn_heads_times_head_dim embed_dim"
            " -> "
            "batch_size seq_len num_attn_heads_times_head_dim",
        )

        k_vec = einsum(
            x, self.key_w.weight,
            "batch_size seq_len embed_dim, "
            "num_attn_heads_times_head_dim embed_dim"
            " -> "
            "batch_size seq_len num_attn_heads_times_head_dim",
        )

        v_vec = einsum(
            x, self.value_w.weight,
            "batch_size seq_len embed_dim, "
            "num_attn_heads_times_head_dim embed_dim"
            " -> "
            "batch_size seq_len num_attn_heads_times_head_dim",
        )


        # -------------------------------------------------
        # 2. Rearrange into multi-head form
        #    (batch_size, seq_len, num_attn_heads * head_dim)
        # →  (batch_size, num_attn_heads, seq_len, head_dim)
        # -------------------------------------------------
        q = rearrange(
            q_vec,
            "batch_size seq_len (num_attn_heads head_dim)"
            " -> "
            "batch_size num_attn_heads seq_len head_dim",
            num_attn_heads=self.num_attn_heads,
        )

        k = rearrange(
            k_vec,
            "batch_size seq_len (num_attn_heads head_dim)"
            " -> "
            "batch_size num_attn_heads seq_len head_dim",
            num_attn_heads=self.num_attn_heads,
        )

        v = rearrange(
            v_vec,
            "batch_size seq_len (num_attn_heads head_dim)"
            " -> "
            "batch_size num_attn_heads seq_len head_dim",
            num_attn_heads=self.num_attn_heads,
        )


        


In [ ]:
model = Level3SingleHeadCausalAttention_Model()
configs = MODEL_CONFIGS

training_engine.train_and_save_model(model, configs, device, 100)
training_engine.plot_loss_history(configs)


In [ ]:
prompt = "there was a"
inference_engine.advanced_inference(model, configs, 20, prompt, device, eos_token_id=83)